# GIN — Fixed Architecture, Batch of Feature Configurations (HI-Small)

- **Question:** what do engineered edge features add when the GNN architecture is held *exactly* fixed?
- **Knobs per config:** `mp` = edge features in message passing (`none` / `base` / `full`), `readout` = edge features at the classifier (`base` / `full` / `gfp`). Direction and temporal sampling are fixed per batch.
- **Everything shared** (model template, training loop, threshold selection, metrics, plots, saving) lives in `gnn_core.py`. This notebook only sets `OPERATOR`, the config batch and paths.
- **Protocol:** threshold swept on validation each epoch, best-val-F1 checkpoint, test scored once; every metric on train / val / test. `invariant_params` must be **67,587** in every run.

## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order.
- Then clone (or pull) the public repo so the committed `gnn_core.py` is the one imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- The only cell that differs between operator notebooks (`GIN_` / `PNA_` / `GAT_`).
- `CONFIGS`: one dict per run of this batch; run numbers refer to `EXPERIMENTS.md`.
- `OUT_ROOT` mirrors `Outputs/GIN/` in the repo; the zip in section 6 is what gets committed (minus `best.pt`).

In [ ]:
OPERATOR = 'gin'

# knob configs of this batch — one dict per run
CONFIGS = [
    dict(mp='none', readout='base'),   # Run 1  — GIN,  baseline features
    dict(mp='base', readout='base'),   # Run 2  — GINE, baseline features
    dict(mp='none', readout='full'),   # Run 7  — GIN,  +GFP at the readout
    dict(mp='base', readout='full'),   # Run 16 — GINE, GFP at the readout only
    dict(mp='full', readout='full'),   # Run 8  — GINE, +GFP everywhere
]
MP_DIRECTION      = 'in'     # 'in' | 'bidirectional' — fixed for the whole batch
TEMPORAL_SAMPLING = False    # True samples only edges earlier than the seed (needs pyg-lib)

# paths
DATA_DIR    = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
OUT_ROOT    = '/kaggle/working/Outputs/GIN'      # mirrors Outputs/GIN/ in the repo

import gnn_core as core
os.makedirs(OUT_ROOT, exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | gnn_core from {core.__file__}')

## 2. Load graphs

- Three cumulative PyG snapshots; context edges carry label −1, only `eval_mask` edges are scored.
- `col_sets` maps each knob value to `edge_attr` columns: base = first 20, gfp = next 61, full = all 81.

In [ ]:
graphs, col_sets, node_dim = core.load_graphs(DATA_DIR)

## 3. Model

- Fixed template (`gnn_core.EdgeClassifier`): `node_proj` → 2 message-passing layers (hidden 128, dropout 0.3, residual) → readout MLP on `[h_src ‖ h_dst ‖ e_seed]`.
- `mp = none` → `GINConv`; otherwise `GINEConv(edge_dim)` adds only the edge projection `W_e`.
- Between configs only `W_e` and the readout's first `Linear` change width; `invariant_params` counts the rest and must be 67,587.

In [ ]:
cfg = CONFIGS[0]
model = core.build_model(OPERATOR, node_dim, len(col_sets[cfg['mp']]), len(col_sets[cfg['readout']]),
                         MP_DIRECTION == 'bidirectional')
print(model)
print(f'params {core.count_params(model):,} | invariant_params {core.invariant_params(model):,}   (config {cfg})')
del model

## 4. Run the batch

- One call per config: seed → model → loaders → 20 epochs → best checkpoint → all splits scored → `results.json`, `history.csv`, `curves.png`, `best.pt`, `predictions.csv` under `OUT_ROOT/<run>/`.
- `predictions.csv`: every evaluated edge of train / val / test with `edge_id` (= row in `Data/edge_features.csv`), `src`, `dst`, `edge_time`, `y`, `prob` (probability) and `pred` (class at the val-chosen threshold) — for the typology / error analyses.
- Per-epoch line: train / val loss, F1 (train at the val threshold), PR-AUC, seconds, RAM; `* new best` marks a checkpoint.

In [ ]:
rows = []
for cfg in CONFIGS:
    print('=' * 90)
    rows.append(core.run_experiment(OPERATOR, cfg, graphs, col_sets, node_dim, OUT_ROOT,
                                    MP_DIRECTION, TEMPORAL_SAMPLING))
print('=' * 90)
print('batch complete')

## 5. Comparison across the batch

- `batch_summary.csv` holds every `<split>_<metric>` column plus `threshold`, `best_epoch`, `params`, `invariant_params`.
- `invariant_params` must be identical in every row (asserted).

In [ ]:
summary = core.summarize_batch(rows, OUT_ROOT)
summary[core.DISPLAY_COLS].round(4)

## 6. Download

- Zip `OUT_ROOT`; extract into `Outputs/GIN/` in the repo and commit the small result files (`best.pt` and `predictions.csv` stay out of git).

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_GIN', 'zip', OUT_ROOT)